In [291]:
%load_ext autoreload
%autoreload 1
%aimport classes.GaloisField
%aimport classes.GolayDecoder

import numpy as np

from classes.GaloisField import *
from classes.GaloisPoly  import *
from classes.GolayEncoder import GolayEncoder
from classes.GolayDecoder import GolayDecoder

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Generate Galois Field

In [292]:
gf              = GaloisField(1,0b11)
encoder_model   = GolayEncoder()
k               = encoder_model._k
n               = encoder_model._n


Field Closed Succesfully!, 1 Non-Zero Elements


## 1. Codewords Test

### Generate all codewords

In [293]:
encoder_output = None
for w in range(2**k):
    w   = gf.do_unpack(w, bit_width=k)
    cw  = encoder_model.encode(w)
    encoder_output = cw if encoder_output is None else np.vstack((encoder_output, cw))

n_codewords = len(encoder_output)

### Decoding of codewords

In [294]:
n_codewords
#encoder_output 
decoder_model = GolayDecoder()

decoder_output = []

for i in range(n_codewords):
    decoder_output.append(decoder_model.correct(encoder_output[i]))

#print(decoder_output)

Field Closed Succesfully!, 1 Non-Zero Elements


### Error (`o_err`)

In [295]:
o_err           = []
corrected       = []
uncorrectable   = []

o_corrected     = []
o_uncorrectable = []

for i in range(n_codewords):
    # 0: i_rx, 1: o_corrected, 2: o_uncorrectable
    o_err.append(decoder_output[i][0] - encoder_output[i])
    # Flags for uvm
    o_corrected.append(int(decoder_output[i][1]))
    o_uncorrectable.append(int(decoder_output[i][2]))
    # Corrected and uncorrectable bool
    corrected.append(decoder_output[i][1])
    uncorrectable.append(decoder_output[i][2])

# print(o_err, o_corrected, o_uncorrectable)

print(np.array(o_err).shape)
print(np.array(o_corrected).shape)
print(np.array(o_uncorrectable).shape)



(4096, 24)
(4096,)
(4096,)


In [296]:
with open("outputs/top_testing/codewords/codewords_testing_golay_code.svh", "w") as file:

    # Codewords
    file.write("\tconstraint golay_code {\n")
    file.write("\t\trx_data inside {\n")

    for i, cw in enumerate(encoder_output):
        v = gf.do_pack(cw)

        if i == len(encoder_output) - 1:
            file.write(f"\t\t\t24'b{v:024b}\t}};\n")
        else:
            file.write(f"\t\t\t24'b{v:024b}\t,\n")


    # Decoded codewords
    file.write("\n\t\tdecoded_data inside {\n")

    for i, cw in enumerate(decoder_output):
        v = gf.do_pack(cw[0])

        if i == len(decoder_output) - 1:
            file.write(f"\t\t\t24'b{v:024b}\t}};\n")
        else:
            file.write(f"\t\t\t24'b{v:024b}\t,\n")

    # Errors
    file.write("\n\t\terr inside {\n")

    for i, err in enumerate(o_err):
        v = gf.do_pack(err)

        if i == len(o_err) - 1:
            file.write(f"\t\t\t24'b{v:024b}\t}};\n")
        else:
            file.write(f"\t\t\t24'b{v:024b}\t,\n")


    # Corrected received words flag
    file.write("\n\t\tcorrected_data inside {\n")

    for i, corrected in enumerate(o_corrected):
        if i == len(o_corrected) - 1:
            file.write(f"\t\t\t1'b{corrected}\t}};\n")
        else:
            file.write(f"\t\t\t1'b{corrected}\t,\n")

    # Uncorrectable received words flag
    file.write("\n\t\tuncorrectable inside {\n")

    for i, uncorrectable in enumerate(o_uncorrectable):
        if i == len(o_uncorrectable) - 1:
            file.write(f"\t\t\t1'b{uncorrectable}\t}};\n")
        else:
            file.write(f"\t\t\t1'b{uncorrectable}\t,\n")


    file.write("\t};\n")

## 2. Codewords with errors Test

In [297]:
encoder_output 

received_msg_with_error = []

n_errors = 5 # 4 errors limit

for n in range(1, n_errors):
    codewords_n_errors = []
    # Select random n positions
    error_positions = np.random.choice(24, n, replace=False)

    for i in range(n_codewords):
        codeword = encoder_output[i].copy()

        # Put errores
        for pos in error_positions:
            codeword[pos] ^= 1

        # Guardar la codeword completa
        codewords_n_errors.append(codeword)

    received_msg_with_error.append(codewords_n_errors)

#print(received_msg_with_error[0][1][3])

# Dimensions

# received_msg_with_error[0] → codewords with 1 error
# received_msg_with_error[1] → codewords with 2 errors
# received_msg_with_error[2] → codewords with 3 errors
# received_msg_with_error[3] → codewords with 4 errors

# received_msg_with_error[0][0] → first codeword with 1 error
# received_msg_with_error[0][1] → second codeword with 1 error
# codeword with 24-bits

# received_msg_with_error[0][1][0] → first bit of codeword with 1 error

# received_msg_with_error[error_group][codeword][bit]
#                          │            │          │
#                          │            │          └── 0 ... 23
#                          │            │
#                          │            └──────────── 0 ... n_codewords-1
#                          │
#                          └───────────────────────── 0 ... 3


### Decoding of codewords with errors

In [298]:
o_no_cw_err = []
o_no_cw_msg = []
o_no_cw_uncorrectable = []
o_no_cw_corrected = []

for n in range(n_errors - 1):

    # Resultados para esta cantidad de errores
    err_n = []
    msg_n = []
    uncorrectable_n = []
    corrected_n = []

    for i in range(n_codewords):
        received = received_msg_with_error[n][i]
        # Decoder
        decoded = decoder_model.correct(received)
        # Syndrome
        s, q = decoder_model.get_s_q(received)
        # Error pattern
        if decoder_model._gf.do_pack(s) != 0:
            err_n.append(decoder_model.get_error(s, q))
        else:
            err_n.append(0)
        # Message/corrected codeword
        msg_n.append(decoded[0])

        # Flags
        uncorrectable_n.append(int(decoded[1]))
        corrected_n.append(int(decoded[2]))

    # Guardar resultados de este número de errores
    o_no_cw_err.append(err_n)
    o_no_cw_msg.append(msg_n)
    o_no_cw_uncorrectable.append(uncorrectable_n)
    o_no_cw_corrected.append(corrected_n)

### Create files `received words with n-errors`

In [299]:
for n in range(n_errors - 1):

    filename = f"outputs/top_testing/error_{n+1}_vectors/codewords_testing_golay_{n+1}_error.svh"

    with open(filename, "w") as file:

        # Codewords received with errors
        file.write("\tconstraint golay_code {\n")
        file.write("\t\trx_data inside {\n")

        for i, cw in enumerate(received_msg_with_error[n]):
            v = gf.do_pack(cw)

            if i == len(received_msg_with_error[n]) - 1:
                file.write(f"\t\t\t24'b{v:024b}\t}};\n")
            else:
                file.write(f"\t\t\t24'b{v:024b}\t,\n")


        # Decoded codewords
        file.write("\n\t\tdecoded_data inside {\n")

        for i, cw in enumerate(o_no_cw_msg[n]):
            v = gf.do_pack(cw)

            if i == len(o_no_cw_msg[n]) - 1:
                file.write(f"\t\t\t24'b{v:024b}\t}};\n")
            else:
                file.write(f"\t\t\t24'b{v:024b}\t,\n")


        # Errors
        file.write("\n\t\terr inside {\n")

        for i, err in enumerate(o_no_cw_err[n]):

            # If there is no error pattern, use a 24-bit zero vector
            if err is None:
                err = [0] * 24

            v = gf.do_pack(err)

            if i == len(o_no_cw_err[n]) - 1:
                file.write(f"\t\t\t24'b{v:024b}\t}};\n")
            else:
                file.write(f"\t\t\t24'b{v:024b}\t,\n")


        # Corrected received words flag
        file.write("\n\t\tcorrected_data inside {\n")

        for i, corrected in enumerate(o_no_cw_corrected[n]):
            if i == len(o_no_cw_corrected[n]) - 1:
                file.write(f"\t\t\t1'b{corrected}\t}};\n")
            else:
                file.write(f"\t\t\t1'b{corrected}\t,\n")


        # Uncorrectable received words flag
        file.write("\n\t\tuncorrectable inside {\n")

        for i, uncorrectable in enumerate(o_no_cw_uncorrectable[n]):
            if i == len(o_no_cw_uncorrectable[n]) - 1:
                file.write(f"\t\t\t1'b{uncorrectable}\t}};\n")
            else:
                file.write(f"\t\t\t1'b{uncorrectable}\t,\n")


        file.write("\t};\n")

## golay (24,12) decoding example

In [300]:
r = encoder_output[3576]
# Generate random error for r (word received/transmitted)
r = r ^ np.array([
    [1,0,0,1,0,0,0,1,0,0,0,0]   ,
    [0,0,0,0,0,0,0,0,0,0,0,0]   ]).flatten()

decoder_model = GolayDecoder()

w, corrected, uncorrectable = decoder_model.decode(r)

# r: word with errors
# word decoded (possible codeword),
# flags (corrected, uncorrectable)
# encoder output (word received transmitted)
r, decoder_model.decode(r), encoder_output[3576]

Field Closed Succesfully!, 1 Non-Zero Elements


(array([0, 1, 0, 0, 1, 1, 1, 0, 1, 0, 0, 0, 0, 0, 1, 1, 0, 1, 1, 0, 1, 1,
        1, 1]),
 (array([1, 1, 0, 1, 1, 1, 1, 1, 1, 0, 0, 0]), True, False),
 array([1, 1, 0, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 1, 1, 0, 1, 1, 0, 1, 1,
        1, 1], dtype=uint8))

In [301]:
# decode all codewords, no errors
for cw in encoder_output:
    w, corrected, uncorrectable = decoder_model.decode(cw, full_codeword=True)
    assert(np.all(w == cw))
    assert(not corrected and not uncorrectable)

for cw in encoder_output:
    error_seed      = np.random.randint(0, 0b11111)
    # decimal error_seed converted into n-bits
    error           = decoder_model._gf.do_unpack(error_seed, bit_width=decoder_model._n)
    # calculate hamming weight
    error_weight    = decoder_model._gf.hamming_weight(error)
    
    np.random.shuffle(error)
    w, corrected, uncorrectable = decoder_model.decode(cw ^ error, full_codeword=True)
    
    # no errors
    if error_weight == 0:
        assert(np.all(w == cw))
        assert(not corrected and not uncorrectable)
    # 1 to 3 errors
    elif 1 <= error_weight <= 3:
        assert(np.all(w == cw))
        assert(corrected and not uncorrectable)
    # 4 errors
    else:
        assert(not np.all(w == cw))
        assert(not corrected and uncorrectable)

    # five or more errors this decoding fails, recovered bits and flags are invalid


## Decoding test with selected values 

Verify the following values:

$$ r_{1} = 0xA5D9A6 $$
$$ r_{2} = 0xA5F9A4 $$
$$ r_{3} = 0xA5C9AA $$

### 1. Obtain values corrected.

In [302]:
import numpy as np

rx_test = [0xA5D9A6, 0xA5F9A4, 0xA5C9AA]

rx_test_array = [
    np.array([int(bit) for bit in format(x, '024b')], dtype=np.uint8)
    for x in rx_test]

rx_test_array

decoded_rx_test        = []
decoded_rx_test_binary = []

for i in range(len(rx_test_array)):
    decoded_rx_test.append(decoder_model.decode(rx_test_array[i], False))

# print(decoded_rx_test)

for decoded, valid, uncorrectable in decoded_rx_test:
    decoded_rx_test_binary.append(
        (decoded, int(valid), int(uncorrectable))
    )

# decoded word, o_corrected, o_uncorrected
decoded_rx_test_binary

[(array([1, 0, 1, 0, 0, 1, 0, 1, 1, 1, 0, 0], dtype=uint8), 1, 0),
 (array([1, 0, 1, 0, 0, 1, 0, 1, 1, 1, 0, 0], dtype=uint8), 1, 0),
 (array([1, 0, 1, 0, 0, 1, 0, 1, 1, 1, 0, 0], dtype=uint8), 0, 1)]

### 2. Obtain mask error

In [303]:
s_q_vectors  = []
error_vector = []

for i in range (len(rx_test_array)):
    s_q_vectors.append(decoder_model.get_s_q(rx_test_array[i]))
    error_vector.append(decoder_model.get_error(s_q_vectors[i][0], s_q_vectors[i][1]))
    if error_vector[i] is None:
        error_vector[i] = [0]*24
    
error_vector

[array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        1, 1], dtype=uint8),
 array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 1], dtype=uint8),
 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]

In [304]:
filename = "outputs/decoding_vectors_test/codewords_testing_golay_test.svh"


with open(filename, "w") as file:

    file.write("\tconstraint golay_code_test_vectors {\n    //0xA5D9A6, 0xA5F9A4, 0xA5C9AA \n")

    # -------------------------------------------------
    # Received codewords
    # -------------------------------------------------
    file.write("\t\trx_data inside {\n")

    for i in range(len(rx_test_array)):

        v = int("".join(map(str, rx_test_array[i])), 2)

        comma = "," if i < len(rx_test_array) - 1 else "};"

        file.write(f"\t\t\t24'b{v:024b}\t{comma}\n")


    # -------------------------------------------------
    # Decoded codewords
    # -------------------------------------------------
    file.write("\n\t\tdecoded_data inside {\n")

    for i in range(len(decoded_rx_test_binary)):

        decoded = decoded_rx_test_binary[i][0]

        v = int("".join(map(str, decoded)), 2)

        comma = "," if i < len(decoded_rx_test_binary) - 1 else "};"

        file.write(f"\t\t\t12'b{v:012b}\t{comma}\n")


    # -------------------------------------------------
    # Error vectors
    # -------------------------------------------------
    file.write("\n\t\terr inside {\n")

    for i in range(len(error_vector)):

        err = error_vector[i]

        v = int("".join(map(str, err)), 2)

        comma = "," if i < len(error_vector) - 1 else "};"

        file.write(f"\t\t\t24'b{v:024b}\t{comma}\n")


    # -------------------------------------------------
    # Corrected flag
    # -------------------------------------------------
    file.write("\n\t\tcorrected_data inside {\n")

    for i in range(len(decoded_rx_test_binary)):

        corrected = decoded_rx_test_binary[i][1]

        comma = "," if i < len(decoded_rx_test_binary) - 1 else "};"

        file.write(f"\t\t\t1'b{corrected}\t{comma}\n")


    # -------------------------------------------------
    # Uncorrectable flag
    # -------------------------------------------------
    file.write("\n\t\tuncorrectable inside {\n")

    for i in range(len(decoded_rx_test_binary)):

        uncorrectable = decoded_rx_test_binary[i][2]

        comma = "," if i < len(decoded_rx_test_binary) - 1 else "};"

        file.write(f"\t\t\t1'b{uncorrectable}\t{comma}\n")


    file.write("\t};\n")